In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl
import os

import matplotlib.pyplot as plt
import seaborn as sns
import corner

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from utils import plot_training_curves, plot_roc_curve, plot_predictions_distribution, set_num_threads, set_seed

In [ ]:
# Limit CPU threads
NUM_CPU = 12
SEED = 42
set_num_threads(NUM_CPU)
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Inputs
BASE_DIR = os.getcwd()
TRAINING_PICKLE = os.path.abspath(f"{BASE_DIR}/from_albert/training_file_kaon_pion.pkl")
OUTPUT_DIR = os.path.abspath(f"{BASE_DIR}/outputs")
OUTPUT_PICKLE = f"{OUTPUT_DIR}/predictions.pkl"
CHECKPOINT_PATH = f"{OUTPUT_DIR}/best_model.pt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training
TEST_SIZE = 0.3
RANDOM_STATE = 42
BATCH_SIZE = 1024
EPOCHS = 10
EXTRA_HIDDEN_SIZE = 5
LR = 1e-3
WEIGHT_DECAY = 0.0
VALIDATE_EVERY = 5
FEATURE_COLS = [
    "Positive_Energy_FCAL",
    "Positive_E1E9_FCAL",
    "Positive_E9E25_FCAL",
    "Positive_SumU_FCAL",
    "Positive_SumV_FCAL",
    "Positive_TrackFCAL_DOCA",
    # --
    "Negative_Energy_FCAL",
    "Negative_E1E9_FCAL",
    "Negative_E9E25_FCAL",
    "Negative_SumU_FCAL",
    "Negative_SumV_FCAL",
    "Negative_TrackFCAL_DOCA",
]

# Core Elements in Supervised Learning

1. Data - input and output pairs we wish to learn a mapping between. For this problem, we are mapping from some detector level variables, i.e. energy and shower shape variables, to some particle id, i.e. pion or kaon.
2. Model - Parameterized mapping from inputs to outputs.
3. Loss Function / Metrics - Quantifies the error between model predictions and data. Not all metrics are appropriate for all problems. Standard example is in disease detection where false negatives are much more costly than false positives.

Optimization - Learning algorithm that trains the model, i.e. backpropagation for neural networks provides gradients which is used by an optimizer like Adam to effectively navigate the parameter space.

# 1. Data

In [ ]:
with open(TRAINING_PICKLE, "rb") as f:
    data = pkl.load(f)

It is important to understand the structure of data. Physicists generally use histograms to visualize the probability density. A **corner** plot can be helpful to visualize pairwise correlations between variables where diagonal subplots are 1D histograms of the variables / features and off-diagonal subplots are 2D histograms of the variables / features.

This allows for quick checks:
- probability densities
- correlations between variables
- outliers
- feature ranges

From the corner plot, it is clear that:
- we need to standardize the features
- a simple cut based analysis can be very difficult to perform here since there is significant overlap between pion and kaon distributions

In [ ]:
feature_cols = [
    'Positive_Energy_FCAL', 'Positive_E1E9_FCAL', 'Positive_E9E25_FCAL',
    'Positive_SumU_FCAL', 'Positive_SumV_FCAL', 'Positive_TrackFCAL_DOCA',
    'Negative_Energy_FCAL', 'Negative_E1E9_FCAL', 'Negative_E9E25_FCAL',
    'Negative_SumU_FCAL', 'Negative_SumV_FCAL', 'Negative_TrackFCAL_DOCA'
]

# Split data by pi_k (label of the dataset)
data_pi = data[data['pi_k'] == 1][feature_cols]
data_k = data[data['pi_k'] == 0][feature_cols]

# Overlay the two distributions
corner.corner(
    data_pi,
    color='k',
    labels=feature_cols,
    show_titles=True,
    hist_kwargs={'density': True, 'alpha': 0.7},
    label_kwargs={'fontsize': 10}
)
_ = corner.corner(
    data_k,
    color='r',
    labels=feature_cols,
    show_titles=True,
    hist_kwargs={'density': True, 'alpha': 0.7},
    label_kwargs={'fontsize': 10},
    fig=plt.gcf()  # overlay on the same figure
)

In [ ]:
# Since original network uses tanh, we need to output onto [-1, 1] range
data = data.rename(columns={"pi_k": "label"})
data['label'] = data['label'] * 2 - 1 # convert to be on tanh range

X = data[FEATURE_COLS].to_numpy(dtype=np.float32)
y = data["label"].to_numpy(dtype=np.float32)

# Split into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)

# use StandardScaler to standardize the features
scaler = StandardScaler()

# fit the scaler to training data only
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# DataLoaders
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).view(-1, 1)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).view(-1, 1)),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# 2. Model Implementation

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_inputs: int):
        super().__init__()
        hidden_size = num_inputs + EXTRA_HIDDEN_SIZE
        self.fc1 = nn.Linear(num_inputs, hidden_size)
        self.fc_out = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc_out(x))
        return x
    
model = MLP(num_inputs=X.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# Model Training

In [ ]:
# Track training history
train_losses, val_losses = [], []
val_aucs = []
val_accs = []

# Train
best_val_auc = -np.inf
for epoch in range(1, EPOCHS + 1):
    
    # Set model to training mode, some layers have different behavior in training and inference
    #   Not true with this model, but good in practice
    model.train()
    
    running_loss, num_samples = 0.0, 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        
        optimizer.zero_grad() # torch accumulates gradients between iterations by default
        
        preds = model(xb)
        loss = criterion(preds, yb)
        
        loss.backward()  # compute gradients
        optimizer.step() # update weights
        
        running_loss += loss.item() * xb.size(0)
        num_samples += xb.size(0)
    avg_train_loss = running_loss / max(1, num_samples)
    train_losses.append(avg_train_loss)

    # Track / log validation metrics intermittently
    if epoch % VALIDATE_EVERY == 0 or epoch == EPOCHS:
        model.eval() # set model to evaluation mode
        with torch.no_grad(): # no need to spend extra compute on gradient computation
            y_true_list, y_pred_list = [], []
            val_loss_sum, val_n = 0.0, 0
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss_sum += loss.item() * xb.size(0)
                val_n += xb.size(0)
                y_true_list.append(yb.squeeze(1).cpu().numpy())
                y_pred_list.append(preds.squeeze(1).cpu().numpy())
            y_true = np.concatenate(y_true_list, axis=0)
            y_pred = np.concatenate(y_pred_list, axis=0)
            y_prob = (y_pred + 1.0) / 2.0
            y_prob_true = ((y_true + 1.0) / 2.0).astype(int)
            try:
                val_auc = roc_auc_score(y_prob_true, y_prob)
            except ValueError:
                val_auc = float("nan")
            val_acc = accuracy_score(y_prob_true, (y_pred >= 0.0).astype(int))
            val_loss = val_loss_sum / max(1, val_n)
            
            # Store validation metrics
            val_losses.append(val_loss)
            val_aucs.append(val_auc)
            val_accs.append(val_acc)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save({"model_state": model.state_dict()}, CHECKPOINT_PATH)

        print(
            f"Epoch {epoch:4d} | train_loss={avg_train_loss:.5f} | val_loss={val_loss:.5f} | val_auc={val_auc:.4f} | val_acc={val_acc:.4f}"
        )

# Plot training curves
plot_training_curves(
    train_losses=train_losses,
    val_losses=val_losses,
    val_aucs=val_aucs,
    val_accs=val_accs,
    save_path=f"{OUTPUT_DIR}/training_curves.png"
)

# Load best and run inference on full set
if os.path.exists(CHECKPOINT_PATH):
    state = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(state["model_state"])
model.eval()
with torch.no_grad():
    feats_tensor = torch.from_numpy(X).float().to(device)
    preds = model(feats_tensor).squeeze(1).cpu().numpy()
    probs = (preds + 1.0) / 2.0

# Plot ROC curve
y_true_binary = ((y + 1.0) / 2.0).astype(int)
plot_roc_curve(
    y_true=y_true_binary,
    y_scores=probs,
    save_path=f"{OUTPUT_DIR}/roc_curve.png"
)

# Plot predictions distribution
plot_predictions_distribution(
    y_true=y_true_binary,
    y_pred=probs,
    save_path=f"{OUTPUT_DIR}/predictions_distribution.png"
)

pred_series = pd.Series(probs, index=df.index, name="prob_signal")
out_path = os.path.join(OUTPUT_DIR, OUTPUT_PICKLE)
pred_series.to_pickle(out_path)
print(f"Saved predictions to {out_path}")